# 🏥 Medical LLM Assistant — Tech Challenge Fase 3

**Autor:** Felipe Huszar  
**Repositório:** https://github.com/felipe-huszar/medical-llm-assistant  

Assistente médico com LLM (Mistral fine-tuned via LoRA), orquestrado por **LangGraph** e memória via **ChromaDB**.

---

## 🔧 Modos de Execução
| Modo | GPU | Descrição |
|------|-----|-----------|
| `USE_MOCK_LLM=true` | ❌ Não precisa | Respostas determinísticas, ideal para testes |
| `USE_MOCK_LLM=false` | ✅ T4 ou superior | Modelo Mistral fine-tuned via LoRA |

---

## 📋 Estrutura do Notebook
1. Clone do repositório
2. Instalação de dependências
3. Configuração de ambiente
4. Execução dos testes (~78 testes)
5. Demonstração do pipeline (4 cenários)
6. Interface Gradio (UI completa)
7. 📤 Publicar logs no Gist (para revisão do assistente)

## 1️⃣ Clone do Repositório

In [ ]:
import os, sys

REPO_URL = "https://github.com/felipe-huszar/medical-llm-assistant.git"
REPO_DIR = "/content/medical-llm-assistant"

if os.path.exists(REPO_DIR):
    print("Repositório já existe — atualizando...")
    !cd {REPO_DIR} && git pull
    # Limpa cache de módulos src/ para pegar código novo
    removed = [sys.modules.pop(k) for k in list(sys.modules) if k.startswith("src.")]
    if removed:
        print(f"{len(removed)} módulos src/ recarregados do cache")
else:
    print("Clonando repositório...")
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
print("Repositorio pronto.")
!ls -la


## 2️⃣ Instalação de Dependências

In [ ]:
import importlib.util

# Verifica chromadb (instalado junto com o resto — nunca vem pré-instalado no Colab)
if importlib.util.find_spec('chromadb') is None:
    print('📦 Instalando dependências...')
    import subprocess
    subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'])
    subprocess.run(['pip', 'install', '-q', 'unsloth', 'peft', 'bitsandbytes', 'accelerate', 'gdown'])
    print('🔄 Reiniciando runtime... Execute esta célula novamente após o restart.')
    import os; os.kill(os.getpid(), 9)
else:
    print('✅ Dependências instaladas.')


## 3️⃣ Configuração de Ambiente

In [ ]:
# Google Drive — montar para acessar modelo treinado
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive montado.')


In [ ]:
import os, sys

# ──────────────────────────────────────────────────────────
# ⬇️  MUDE AQUI para alternar entre MockLLM e modelo real
# ──────────────────────────────────────────────────────────
USE_MOCK = 'false'   # 'false' para usar Mistral LoRA (requer GPU T4)

# Caminho do modelo treinado no seu Drive
SEU_MODELO_ZIP = '/content/drive/MyDrive/FIAP-TC-Fase3/medical_assistant_llm.zip'
LORA_LOCAL_PATH = '/content/model'

os.environ['USE_MOCK_LLM']  = USE_MOCK
os.environ['MODEL_PATH']    = LORA_LOCAL_PATH
os.environ['BASE_MODEL_ID'] = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit'
os.environ['CHROMA_DB_PATH']= '/content/chroma_db'

sys.path.insert(0, '/content/medical-llm-assistant')

print('=' * 55)
print('  🔧  CONFIGURAÇÃO ATUAL')
print('=' * 55)
print(f"  USE_MOCK_LLM   = {os.environ['USE_MOCK_LLM']}")
print(f"  MODEL_PATH     = {os.environ['MODEL_PATH']}")
print(f"  BASE_MODEL_ID  = {os.environ['BASE_MODEL_ID']}")
print(f"  CHROMA_DB_PATH = {os.environ['CHROMA_DB_PATH']}")
print('=' * 55)

# Download automático do adapter quando modo real
if USE_MOCK.lower() != 'true':
    import subprocess
    print('\n📥 Baixando adapter LoRA do Drive do Lucas...')
    # Extrai ZIP do seu modelo
    import zipfile
    print(f'📦 Extraindo: {SEU_MODELO_ZIP}')
    with zipfile.ZipFile(SEU_MODELO_ZIP, 'r') as zf:
        zf.extractall(LORA_LOCAL_PATH)
    print('✅ Modelo extraído.')
    result = type('obj', (object,), {'returncode': 0, 'stdout': '', 'stderr': ''})()
    print(result.stdout)
    if result.returncode != 0:
        print('⚠️  gdown stderr:', result.stderr)
   

    # Valida (inclusive subpastas)
    cfg = None
    wts = None
    for root, _, files in os.walk(LORA_LOCAL_PATH):
        if 'adapter_config.json' in files and 'adapter_model.safetensors' in files:
            cfg = os.path.join(root, 'adapter_config.json')
            wts = os.path.join(root, 'adapter_model.safetensors')
            break

    if cfg and wts:
        final_model_path = os.path.dirname(cfg)
        os.environ['MODEL_PATH'] = final_model_path
        print('✅ Adapter LoRA baixado e validado!')
        print('📁 MODEL_PATH final =', os.environ['MODEL_PATH'])
    else:
        print('❌ Falha no download. Verifique se a pasta do Drive contém adapter_config.json e adapter_model.safetensors.')
        print('   Caminho esperado:', SEU_MODELO_ZIP)
else:
    print('\nℹ️  Modo Mock ativo — adapter não necessário.')


## ⚠️ Limpeza do ChromaDB

> **ATENÇÃO:** Isso apaga **todos** os pacientes e históricos. Execute apenas se tiver certeza.


In [ ]:
# ⚠️ LIMPEZA TOTAL DO CHROMADB — execute apenas se tiver certeza!
CONFIRMAR = False  # mude para True para confirmar

if CONFIRMAR:
    import src.rag.patient_rag as rag
    # Limpa via API (sem restart necessário)
    client = rag._get_client()
    for col_name in ['patients', 'consultations']:
        try:
            client.delete_collection(col_name)
            print(f'✅ Collection {col_name!r} removida')
        except Exception:
            print(f'ℹ️  Collection {col_name!r} não existia')
    # Reseta o cache do cliente para forçar recriação
    rag._client = None
    rag._client_path = None
    # Re-seed
    from src.rag.patient_rag import seed_from_file
    n = seed_from_file('data/patients_seed.json')
    print(f'✅ {n} pacientes seed reinseridos. ChromaDB limpo e pronto!')
else:
    print('⚠️  Limpeza NÃO executada. Mude CONFIRMAR = True para prosseguir.')


## 4️⃣ Execução dos Testes

In [ ]:
# ── Testes Unitários (Safety Gate)
print('🧪 Testes Unitários — Safety Gate')
print('-' * 55)
!python -m pytest tests/unit -v --tb=short

In [ ]:
# ── Testes de Integração (Pipeline LangGraph)
print('🧪 Testes de Integração — Pipeline')
print('-' * 55)
!python -m pytest tests/integration -v --tb=short

In [ ]:
# ── Testes E2E (Jornadas Completas + Casos Extendidos)
print('🧪 Testes E2E — Jornadas Completas')
print('-' * 55)
!python -m pytest tests/e2e -v --tb=short

In [ ]:
# ── Sumário consolidado de todos os testes
print('📊 SUMÁRIO COMPLETO')
print('-' * 55)
!python -m pytest tests/ --tb=no -q

## 5️⃣ Demonstração do Pipeline

In [ ]:
from src.graph.pipeline import run_consultation
from src.rag.patient_rag import seed_from_file, get_consultation_history

n = seed_from_file('data/patients_seed.json')
print(f'✅ Pacientes seed carregados: {n}')
print('   (Maria Silva, João Pereira, Ana Costa)')

In [ ]:
# ── Cenário 1: Sintomas Gastrointestinais
print('🩺 CONSULTA 1 — Sintomas Gastrointestinais')
print('Paciente: Maria Silva (CPF 123.456.789-00)')
print('=' * 55)

r1 = run_consultation(
    cpf='123.456.789-00',
    doctor_question=(
        'Paciente relata dores abdominais ao evacuar há 3 semanas, '
        'com sangue nas fezes ocasionalmente. '
        'Quais diagnósticos e exames considerar?'
    ),
)
print(f"Safety: {'✅ Aprovado' if r1['safety_passed'] else '⚠️ Escalado'}  |  "
      f"Escalação: {r1['needs_escalation']}")
print()
print(r1['final_answer'])

In [ ]:
# ── Cenário 2: Retorno com Histórico
print('🩺 CONSULTA 2 — Retorno (mesmo paciente, com histórico)')
print('=' * 55)

r2 = run_consultation(
    cpf='123.456.789-00',
    doctor_question=(
        'Retorno: colonoscopia revelou pólipos adenomatosos. '
        'Dor persiste. Quais próximos passos?'
    ),
)
hist = get_consultation_history('123.456.789-00')
print(f"Safety: {'✅ Aprovado' if r2['safety_passed'] else '⚠️ Escalado'}  |  "
      f"Histórico acumulado: {len(hist)} consulta(s)")
print()
print(r2['final_answer'])

In [ ]:
# ── Cenário 3: Sintomas Cardiovasculares
print('🩺 CONSULTA 3 — Dor Torácica')
print('Paciente: João Pereira (CPF 987.654.321-00)')
print('=' * 55)

r3 = run_consultation(
    cpf='987.654.321-00',
    doctor_question=(
        'Dor torácica em aperto com irradiação para braço esquerdo, '
        'dispneia ao esforço. ECG com supra de ST em V3-V5.'
    ),
)
print(f"Safety: {'✅ Aprovado' if r3['safety_passed'] else '⚠️ Escalado'}")
print()
print(r3['final_answer'])

In [ ]:
# ── Cenário 4: Teste de Segurança — Prescrição Bloqueada
print('🛡️ CENÁRIO 4 — Safety Gate: Tentativa de Prescrição')
print('=' * 55)

from unittest.mock import MagicMock
import json

mock_llm = MagicMock()
mock_llm.invoke.return_value = json.dumps({
    'possible_diagnoses': ['Infecção bacteriana'],
    'recommended_exams': ['Hemocultura'],
    'reasoning': 'Prescrever amoxicilina 500mg 8/8h.',
    'sources': ['Protocolo'],
    'confidence': 0.85,
    'recommendation_type': 'prescription',
})

r4 = run_consultation(
    cpf='SAFETY.001',
    doctor_question='Prescreva antibiótico para infecção.',
    patient_profile={'nome': 'Safety Test', 'idade': 40, 'sexo': 'M', 'peso': 75},
    llm=mock_llm,
)
print(f"Safety: {'✅ Aprovado' if r4['safety_passed'] else '⚠️ BLOQUEADO — escalação ativada'}")
print(f"Prescrição bloqueada: {r4['needs_escalation']}")
print()
print(r4['final_answer'])

## 6️⃣ Interface Gradio (UI Completa)

Clique no link público gerado para abrir a interface.

**Abas disponíveis:**
- `👤 Paciente` — Busca/cadastro por CPF
- `🩺 Consulta` — Pergunta clínica + resposta da IA

In [ ]:
# Carrega o modelo UMA vez e guarda no cache da factory
# A celula de Gradio reutiliza esse cache sem recarregar
import os, src.llm.factory as _factory

if os.environ.get('USE_MOCK_LLM', 'true').lower() != 'true':
    if _factory._cached_llm is None:
        print('Carregando modelo na memoria (unica vez)...')
        _factory.get_llm()  # popula _cached_llm internamente
        print('Modelo pronto e em cache.')
    else:
        print('Modelo ja estava em cache, sem recarga.')
else:
    print('Modo Mock ativo, modelo nao necessario.')


In [ ]:
import sys

# Remove app e src/ do cache — EXCETO src.llm.* que contem o modelo em VRAM
# Apagar src.llm.factory limparia _cached_llm e forcaria reload do modelo
for mod in list(sys.modules):
    if mod == "app" or (mod.startswith("src.") and not mod.startswith("src.llm")):
        del sys.modules[mod]

import app as _app
from src.llm.factory import get_llm

# Reutiliza modelo ja em VRAM (cache em src.llm.factory._cached_llm)
_app.set_llm(get_llm())
print("Modelo configurado no app.")

_app.demo.queue().launch(share=True, debug=True).queue().launch(share=True)


## 7️⃣ 📤 Publicar Logs no Gist

Roda essa célula após executar os testes.  
Ela captura todos os outputs e publica num **Gist secreto** — só quem tem o link acessa.  
Cole o URL gerado no Discord para o assistente revisar.

> **Pré-requisito:** Adicione o secret `GITHUB_PAT` em **🔑 Secrets** (barra lateral esquerda)  
> com scope `gist` apenas — sem acesso a repositórios.

In [ ]:
import subprocess, urllib.request, json, datetime
from google.colab import userdata

# ── Lê o PAT do Colab Secrets (nunca aparece no código)
try:
    PAT = userdata.get('GITHUB_PAT')
except Exception:
    raise RuntimeError('❌ Secret GITHUB_PAT não encontrado. Adicione em 🔑 Secrets.')

# ── ID do Gist fixo (atualiza sempre o mesmo — não cria novos)
GIST_ID = 'f3e6e10d65eff30560abf756467d8d1b'

# ── Captura output de todos os testes
print('🧪 Rodando suite completa para capturar logs...')
result = subprocess.run(
    ['python', '-m', 'pytest', 'tests/', '-v', '--tb=short', '--no-header'],
    capture_output=True, text=True, cwd='/content/medical-llm-assistant'
)
test_output = result.stdout + ('\n--- STDERR ---\n' + result.stderr if result.stderr.strip() else '')

# ── Metadata do run
timestamp = datetime.datetime.utcnow().strftime('%Y-%m-%d %H:%M UTC')
status = '✅ PASSED' if result.returncode == 0 else '❌ FAILED'
header = f"""Medical LLM Assistant — Test Run
Timestamp : {timestamp}
Status    : {status}
Exit code : {result.returncode}
{'='*60}
"""
full_log = header + test_output

# ── Atualiza o Gist existente (PATCH) em vez de criar um novo (POST)
gh_headers = {
    'Authorization': f'token {PAT}',
    'Accept': 'application/vnd.github.v3+json',
    'Content-Type': 'application/json',
}

payload = json.dumps({
    'description': f'Medical LLM — Test Run {timestamp}',
    'files': {
        'test_output.txt': {'content': full_log}
    }
}).encode()

req = urllib.request.Request(
    f'https://api.github.com/gists/{GIST_ID}',
    data=payload,
    headers=gh_headers,
    method='PATCH'
)

with urllib.request.urlopen(req) as r:
    gist = json.loads(r.read())

gist_url = gist['html_url']

print()
print('=' * 60)
print(f'  {status}')
print('=' * 60)
print(f'  📋 Gist atualizado (mesmo link sempre):')
print(f'  👉 {gist_url}')
print()
print('  Cole esse URL no Discord para o assistente revisar.')
print('=' * 60)

## 8️⃣ 📋 Publicar Audit Log no Gist

Rode após usar o Gradio para visualizar a trilha de auditoria das consultas.


In [ ]:
# ── Publicar Audit Log no Gist
# Rode após executar consultas na interface Gradio ou células de demo
import urllib.request, json as _json, os
from google.colab import userdata

AUDIT_LOG_PATH = '/tmp/medical_audit.jsonl'
GIST_ID = 'f3e6e10d65eff30560abf756467d8d1b'  # mesmo Gist fixo

try:
    PAT = userdata.get('GITHUB_PAT')
except Exception:
    PAT = os.environ.get('GITHUB_PAT', '')

if not PAT:
    print('❌ GITHUB_PAT não configurado nos Colab Secrets')
elif not os.path.exists(AUDIT_LOG_PATH):
    print(f'ℹ️  Nenhum audit log encontrado em {AUDIT_LOG_PATH}')
    print('   Rode o Gradio e faça algumas consultas primeiro.')
else:
    with open(AUDIT_LOG_PATH) as f:
        audit_content = f.read()

    lines = [l for l in audit_content.strip().split('\n') if l]
    print(f'📋 {len(lines)} eventos de auditoria encontrados')

    import datetime
    ts = datetime.datetime.utcnow().strftime('%Y-%m-%d %H:%M UTC')
    data = _json.dumps({
        'description': f'Medical LLM — Audit Log {ts}',
        'public': False,
        'files': {'audit_log.jsonl': {'content': audit_content or '(vazio)'}}
    }).encode()

    req = urllib.request.Request(
        f'https://api.github.com/gists/{GIST_ID}',
        data=data,
        headers={'Authorization': f'token {PAT}', 'Content-Type': 'application/json'},
        method='PATCH'
    )
    with urllib.request.urlopen(req) as r:
        gist = _json.loads(r.read())
    print(f'✅ Audit log publicado: {gist["html_url"]}')
    print(f'   Arquivo: audit_log.jsonl ({len(lines)} eventos)')
